# Module 3.2: Multivariate Gaussian HMMs

## Learning Objectives
By completing this notebook, you will:
1. Extend univariate Gaussian HMMs to multivariate observations
2. Understand covariance structures (full, diagonal, spherical)
3. Model multiple assets jointly with regime switching
4. Implement EM algorithm for multivariate parameter estimation
5. Analyze cross-asset correlations in different regimes

## Prerequisites
- Univariate Gaussian HMM (Module 3.1)
- Multivariate Gaussian distributions
- Linear algebra (matrix operations, decompositions)

## Estimated Time: 60 minutes

---

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from scipy.linalg import inv, det
from typing import Tuple, List

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Multivariate Gaussian Emissions

### Univariate Recap
For scalar observations: $P(o_t | q_t = i) = \mathcal{N}(o_t; \mu_i, \sigma_i^2)$

### Multivariate Extension
For D-dimensional observations: $\mathbf{o}_t \in \mathbb{R}^D$

$$P(\mathbf{o}_t | q_t = i) = \mathcal{N}(\mathbf{o}_t; \boldsymbol{\mu}_i, \boldsymbol{\Sigma}_i)$$

Where:
- $\boldsymbol{\mu}_i \in \mathbb{R}^D$: Mean vector for state i
- $\boldsymbol{\Sigma}_i \in \mathbb{R}^{D \times D}$: Covariance matrix (positive definite)

### PDF Formula

$$P(\mathbf{o}_t | q_t = i) = \frac{1}{(2\pi)^{D/2}|\boldsymbol{\Sigma}_i|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{o}_t - \boldsymbol{\mu}_i)^T \boldsymbol{\Sigma}_i^{-1}(\mathbf{o}_t - \boldsymbol{\mu}_i)\right)$$

### Log-Likelihood

$$\log P(\mathbf{o}_t | q_t = i) = -\frac{D}{2}\log(2\pi) - \frac{1}{2}\log|\boldsymbol{\Sigma}_i| - \frac{1}{2}(\mathbf{o}_t - \boldsymbol{\mu}_i)^T \boldsymbol{\Sigma}_i^{-1}(\mathbf{o}_t - \boldsymbol{\mu}_i)$$

## 2. Covariance Structure Types

Different constraints on $\boldsymbol{\Sigma}_i$ balance flexibility vs. parameters:

### Full Covariance
- $\boldsymbol{\Sigma}_i$: Unrestricted positive definite
- Parameters: $D(D+1)/2$ per state
- Most flexible, captures all correlations

### Diagonal Covariance
- $\boldsymbol{\Sigma}_i = \text{diag}(\sigma_{i1}^2, ..., \sigma_{iD}^2)$
- Parameters: D per state
- Assumes independence within state

### Spherical Covariance
- $\boldsymbol{\Sigma}_i = \sigma_i^2 \mathbf{I}$
- Parameters: 1 per state
- Isotropic noise

### Tied Covariance
- $\boldsymbol{\Sigma}_i = \boldsymbol{\Sigma}$ for all i
- Parameters: $D(D+1)/2$ total
- Shared across states

In [ ]:
# Purpose: Implement multivariate Gaussian HMM
# Key Concept: Covariance captures cross-asset dependencies

class MultivariateGaussianHMM:
    """
    Multivariate Gaussian HMM with various covariance types.
    
    Parameters
    ----------
    n_states : int
        Number of hidden states
    n_features : int
        Dimension of observations
    covariance_type : str
        'full', 'diag', 'spherical', or 'tied'
    """
    
    def __init__(self, n_states, n_features, covariance_type='full'):
        self.K = n_states
        self.D = n_features
        self.covariance_type = covariance_type
        
        # Initialize parameters randomly
        self.pi = np.ones(self.K) / self.K
        self.A = np.random.rand(self.K, self.K)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        
        self.means = np.random.randn(self.K, self.D) * 0.001
        
        # Initialize covariances based on type
        if covariance_type == 'full':
            self.covars = np.array([np.eye(self.D) * 0.01 for _ in range(self.K)])
        elif covariance_type == 'diag':
            self.covars = np.ones((self.K, self.D)) * 0.01
        elif covariance_type == 'spherical':
            self.covars = np.ones(self.K) * 0.01
        elif covariance_type == 'tied':
            self.covars = np.eye(self.D) * 0.01
    
    def _get_covariance(self, state):
        """Get full covariance matrix for a state."""
        if self.covariance_type == 'full':
            return self.covars[state]
        elif self.covariance_type == 'diag':
            return np.diag(self.covars[state])
        elif self.covariance_type == 'spherical':
            return np.eye(self.D) * self.covars[state]
        elif self.covariance_type == 'tied':
            return self.covars
    
    def _emission_log_prob(self, obs, state):
        """
        Compute log P(obs | state).
        
        Parameters
        ----------
        obs : ndarray (D,) or (T, D)
            Observation(s)
        state : int
            State index
        
        Returns
        -------
        log_prob : float or ndarray
            Log probability
        """
        mean = self.means[state]
        covar = self._get_covariance(state)
        
        # Use scipy for numerical stability
        return stats.multivariate_normal.logpdf(obs, mean=mean, cov=covar)
    
    def _emission_prob(self, obs, state):
        """Compute P(obs | state)."""
        return np.exp(self._emission_log_prob(obs, state))
    
    def forward(self, observations):
        """Forward algorithm for multivariate observations."""
        T = len(observations)
        alpha = np.zeros((T, self.K))
        scaling = np.zeros(T)
        
        # Initialize
        for i in range(self.K):
            alpha[0, i] = self.pi[i] * self._emission_prob(observations[0], i)
        scaling[0] = np.sum(alpha[0])
        alpha[0] /= scaling[0]
        
        # Iterate
        for t in range(1, T):
            for j in range(self.K):
                alpha[t, j] = np.sum(alpha[t-1] * self.A[:, j]) * \
                              self._emission_prob(observations[t], j)
            scaling[t] = np.sum(alpha[t])
            if scaling[t] > 0:
                alpha[t] /= scaling[t]
        
        log_likelihood = np.sum(np.log(scaling + 1e-300))
        return alpha, scaling, log_likelihood
    
    def backward(self, observations, scaling):
        """Backward algorithm."""
        T = len(observations)
        beta = np.zeros((T, self.K))
        beta[-1] = 1.0 / scaling[-1]
        
        for t in range(T-2, -1, -1):
            for i in range(self.K):
                beta[t, i] = np.sum(
                    self.A[i, :] *
                    np.array([self._emission_prob(observations[t+1], j) 
                             for j in range(self.K)]) *
                    beta[t+1, :]
                )
            if scaling[t] > 0:
                beta[t] /= scaling[t]
        
        return beta
    
    def fit(self, observations, max_iter=50, tol=1e-4, verbose=True):
        """
        Fit HMM using Baum-Welch (EM) algorithm.
        
        Parameters
        ----------
        observations : ndarray (T, D)
            Observation sequences
        """
        T = len(observations)
        log_likelihoods = []
        
        for iteration in range(max_iter):
            # E-step
            alpha, scaling, log_lik = self.forward(observations)
            beta = self.backward(observations, scaling)
            
            # Compute gamma and xi
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True)
            
            xi = np.zeros((T-1, self.K, self.K))
            for t in range(T-1):
                denom = 0.0
                for i in range(self.K):
                    for j in range(self.K):
                        xi[t, i, j] = alpha[t, i] * self.A[i, j] * \
                                      self._emission_prob(observations[t+1], j) * \
                                      beta[t+1, j]
                        denom += xi[t, i, j]
                if denom > 0:
                    xi[t] /= denom
            
            # M-step
            # Update pi
            self.pi = gamma[0]
            
            # Update A
            for i in range(self.K):
                for j in range(self.K):
                    numer = np.sum(xi[:, i, j])
                    denom = np.sum(gamma[:-1, i])
                    self.A[i, j] = numer / (denom + 1e-10)
            self.A = self.A / self.A.sum(axis=1, keepdims=True)
            
            # Update means
            for i in range(self.K):
                gamma_sum = np.sum(gamma[:, i])
                self.means[i] = np.sum(gamma[:, i, np.newaxis] * observations, axis=0) / (gamma_sum + 1e-10)
            
            # Update covariances
            if self.covariance_type == 'full':
                for i in range(self.K):
                    gamma_sum = np.sum(gamma[:, i])
                    diff = observations - self.means[i]
                    self.covars[i] = (gamma[:, i, np.newaxis, np.newaxis] * 
                                     diff[:, :, np.newaxis] * diff[:, np.newaxis, :]).sum(axis=0)
                    self.covars[i] /= (gamma_sum + 1e-10)
                    # Add regularization
                    self.covars[i] += np.eye(self.D) * 1e-6
            elif self.covariance_type == 'diag':
                for i in range(self.K):
                    gamma_sum = np.sum(gamma[:, i])
                    diff = observations - self.means[i]
                    self.covars[i] = np.sum(gamma[:, i, np.newaxis] * diff**2, axis=0) / (gamma_sum + 1e-10)
                    self.covars[i] += 1e-6  # Regularization
            
            log_likelihoods.append(log_lik)
            
            if verbose and (iteration % 10 == 0 or iteration == max_iter - 1):
                print(f"Iteration {iteration:3d}: log L = {log_lik:.4f}")
            
            if iteration > 0 and abs(log_likelihoods[-1] - log_likelihoods[-2]) < tol:
                if verbose:
                    print(f"Converged after {iteration+1} iterations")
                break
        
        return log_likelihoods
    
    def predict(self, observations):
        """Viterbi decoding."""
        T = len(observations)
        log_delta = np.zeros((T, self.K))
        psi = np.zeros((T, self.K), dtype=int)
        
        # Initialize
        for i in range(self.K):
            log_delta[0, i] = np.log(self.pi[i] + 1e-300) + \
                             self._emission_log_prob(observations[0], i)
        
        # Iterate
        for t in range(1, T):
            for j in range(self.K):
                probs = log_delta[t-1] + np.log(self.A[:, j] + 1e-300)
                psi[t, j] = np.argmax(probs)
                log_delta[t, j] = probs[psi[t, j]] + \
                                 self._emission_log_prob(observations[t], j)
        
        # Backtrack
        states = np.zeros(T, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        
        return states

print("Multivariate Gaussian HMM class loaded successfully.")
print(f"Supported covariance types: full, diag, spherical, tied")

## 3. Two-Asset Portfolio Example

Model joint returns of two assets with regime-switching correlation.

In [ ]:
# Purpose: Generate synthetic two-asset data with regime switching
# Key Concept: Correlation structure changes across regimes

def generate_two_asset_data(T=1000, random_seed=42):
    """
    Generate two-asset returns with 3 regimes:
    - Bull: Positive returns, positive correlation
    - Bear: Negative returns, high positive correlation (flight to quality)
    - Sideways: Low returns, low correlation
    """
    np.random.seed(random_seed)
    
    # Define regimes
    regime_params = [
        {  # Bull
            'mean': np.array([0.0008, 0.0006]),
            'cov': np.array([[0.0001, 0.00005],
                            [0.00005, 0.00008]])
        },
        {  # Bear
            'mean': np.array([-0.0010, -0.0012]),
            'cov': np.array([[0.0004, 0.00030],
                            [0.00030, 0.0005]])
        },
        {  # Sideways
            'mean': np.array([0.0001, 0.0002]),
            'cov': np.array([[0.00015, 0.00002],
                            [0.00002, 0.00012]])
        }
    ]
    
    # Transition matrix
    A = np.array([
        [0.95, 0.03, 0.02],
        [0.05, 0.90, 0.05],
        [0.10, 0.05, 0.85]
    ])
    
    pi = np.array([0.5, 0.2, 0.3])
    
    # Simulate states
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(3, p=pi)
    
    for t in range(1, T):
        states[t] = np.random.choice(3, p=A[states[t-1]])
    
    # Generate observations
    observations = np.zeros((T, 2))
    for t in range(T):
        regime = states[t]
        observations[t] = np.random.multivariate_normal(
            regime_params[regime]['mean'],
            regime_params[regime]['cov']
        )
    
    return observations, states

# Generate data
returns, true_states = generate_two_asset_data(T=1000)

print("Two-Asset Portfolio Data")
print("="*70)
print(f"Shape: {returns.shape}")
print(f"\nAsset 1 statistics:")
print(f"  Mean: {returns[:, 0].mean():.6f}")
print(f"  Std:  {returns[:, 0].std():.6f}")
print(f"\nAsset 2 statistics:")
print(f"  Mean: {returns[:, 1].mean():.6f}")
print(f"  Std:  {returns[:, 1].std():.6f}")
print(f"\nCorrelation: {np.corrcoef(returns.T)[0, 1]:.4f}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative returns
ax1 = axes[0, 0]
ax1.plot(np.cumsum(returns[:, 0]), label='Asset 1', alpha=0.7)
ax1.plot(np.cumsum(returns[:, 1]), label='Asset 2', alpha=0.7)
ax1.set_xlabel('Time')
ax1.set_ylabel('Cumulative Return')
ax1.set_title('Cumulative Returns')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Scatter plot
ax2 = axes[0, 1]
colors = ['green', 'red', 'gray']
labels = ['Bull', 'Bear', 'Sideways']
for regime in range(3):
    mask = true_states == regime
    ax2.scatter(returns[mask, 0], returns[mask, 1], 
               c=colors[regime], label=labels[regime], alpha=0.4, s=20)
ax2.set_xlabel('Asset 1 Return')
ax2.set_ylabel('Asset 2 Return')
ax2.set_title('Joint Return Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# True states
ax3 = axes[1, 0]
ax3.fill_between(range(len(true_states)), true_states, alpha=0.7, step='mid')
ax3.set_xlabel('Time')
ax3.set_ylabel('Regime')
ax3.set_title('True Hidden States')
ax3.set_yticks([0, 1, 2])
ax3.set_yticklabels(labels)
ax3.grid(True, alpha=0.3)

# Rolling correlation
ax4 = axes[1, 1]
window = 50
rolling_corr = pd.Series(returns[:, 0]).rolling(window).corr(pd.Series(returns[:, 1]))
ax4.plot(rolling_corr, linewidth=1.5)
ax4.set_xlabel('Time')
ax4.set_ylabel('Rolling Correlation')
ax4.set_title(f'Rolling Correlation (window={window})')
ax4.axhline(0, color='black', linestyle='--', alpha=0.3)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Fit Multivariate HMM

Estimate parameters from the two-asset data.

In [ ]:
# Purpose: Fit multivariate Gaussian HMM to portfolio data
# Key Concept: Jointly model means and covariances

print("Fitting 3-state Multivariate Gaussian HMM (full covariance)")
print("="*70 + "\n")

hmm = MultivariateGaussianHMM(n_states=3, n_features=2, covariance_type='full')
log_liks = hmm.fit(returns, max_iter=100, verbose=True)

print("\n" + "="*70)
print("LEARNED PARAMETERS")
print("="*70)

for state in range(3):
    print(f"\nState {state}:")
    print(f"  Mean vector:")
    print(f"    Asset 1: {hmm.means[state, 0]:.6f}")
    print(f"    Asset 2: {hmm.means[state, 1]:.6f}")
    
    cov = hmm._get_covariance(state)
    corr = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    
    print(f"  Covariance matrix:")
    print(f"    {cov}")
    print(f"  Correlation: {corr:.4f}")
    print(f"  Volatilities: [{np.sqrt(cov[0,0]):.4f}, {np.sqrt(cov[1,1]):.4f}]")

print("\nTransition matrix:")
print(hmm.A)

# Decode states
decoded_states = hmm.predict(returns)

print("\n" + "="*70)
print("REGIME IDENTIFICATION")
print("="*70)

# Label states by mean return
state_means = hmm.means.mean(axis=1)
sorted_idx = np.argsort(state_means)
label_map = {sorted_idx[2]: 'Bull', sorted_idx[1]: 'Sideways', sorted_idx[0]: 'Bear'}

for state in range(3):
    mask = decoded_states == state
    print(f"\nState {state} ({label_map[state]}):")
    print(f"  Days: {mask.sum()}")
    print(f"  Asset 1 mean: {returns[mask, 0].mean():.6f}")
    print(f"  Asset 2 mean: {returns[mask, 1].mean():.6f}")
    print(f"  Empirical correlation: {np.corrcoef(returns[mask].T)[0, 1]:.4f}")

## 5. Regime-Specific Correlation Analysis

Visualize how correlation structure changes across regimes.

In [ ]:
# Purpose: Analyze correlation dynamics across regimes
# Key Concept: Crisis periods show correlation breakdown

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

colors = ['green', 'red', 'gray']
labels = [label_map[i] for i in range(3)]

# Row 1: Scatter plots by regime
for state in range(3):
    ax = axes[0, state]
    mask = decoded_states == state
    
    ax.scatter(returns[mask, 0], returns[mask, 1], 
              c=colors[state], alpha=0.5, s=20)
    
    # Add ellipse for learned distribution
    mean = hmm.means[state]
    cov = hmm._get_covariance(state)
    
    # Eigenvalues for ellipse
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    
    from matplotlib.patches import Ellipse
    for n_std in [1, 2]:
        width, height = 2 * n_std * np.sqrt(eigenvalues)
        ellipse = Ellipse(mean, width, height, angle=angle,
                         facecolor='none', edgecolor='black', 
                         linewidth=2, linestyle='--')
        ax.add_patch(ellipse)
    
    ax.set_xlabel('Asset 1 Return')
    ax.set_ylabel('Asset 2 Return')
    ax.set_title(f'{labels[state]} Regime')
    ax.grid(True, alpha=0.3)
    
    corr = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    ax.text(0.05, 0.95, f'ρ = {corr:.3f}', 
           transform=ax.transAxes, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Row 2: Time series with regime overlay
ax_ts = axes[1, :]
ax_ts = plt.subplot(2, 1, 2)

# Plot returns
ax_ts.plot(returns[:, 0], label='Asset 1', alpha=0.7, linewidth=0.8)
ax_ts.plot(returns[:, 1], label='Asset 2', alpha=0.7, linewidth=0.8)

# Color background by regime
for i in range(len(decoded_states)-1):
    color = colors[decoded_states[i]]
    ax_ts.axvspan(i, i+1, alpha=0.1, color=color)

ax_ts.set_xlabel('Time')
ax_ts.set_ylabel('Return')
ax_ts.set_title('Returns with Decoded Regimes')
ax_ts.legend(loc='upper right')
ax_ts.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 1: Covariance Type Comparison

**Task:** Fit HMMs with different covariance types and compare:
1. Log-likelihoods
2. Number of parameters
3. BIC scores

Compute BIC = -2 * log L + k * log(T), where k is number of parameters.

**Expected Output:** Comparison table and best model selection.

In [ ]:
# YOUR CODE HERE
# ---------------

def compare_covariance_types(returns, covariance_types=['full', 'diag', 'spherical']):
    """
    Compare HMM performance with different covariance structures.
    
    Returns
    -------
    results : dict
        Dictionary with log-likelihood, BIC, n_params for each type
    """
    # YOUR IMPLEMENTATION HERE
    # For each covariance type:
    #   - Fit HMM
    #   - Compute log-likelihood
    #   - Count parameters
    #   - Compute BIC
    
    return None

comparison_results = compare_covariance_types(returns)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert comparison_results is not None, "Don't forget to run comparison!"
    
    for cov_type in ['full', 'diag', 'spherical']:
        assert cov_type in comparison_results, f"Missing {cov_type}"
        assert 'log_likelihood' in comparison_results[cov_type]
        assert 'bic' in comparison_results[cov_type]
        assert 'n_params' in comparison_results[cov_type]
    
    # Full should have highest log-likelihood (most flexible)
    full_ll = comparison_results['full']['log_likelihood']
    diag_ll = comparison_results['diag']['log_likelihood']
    assert full_ll >= diag_ll, "Full covariance should fit better"
    
    print("✅ Exercise 1 passed!")
    print("\nCovariance Type Comparison:")
    print("="*70)
    print(f"{'Type':<12s} {'Log-Lik':>12s} {'N Params':>10s} {'BIC':>12s}")
    print("-"*70)
    
    for cov_type in ['full', 'diag', 'spherical']:
        res = comparison_results[cov_type]
        print(f"{cov_type:<12s} {res['log_likelihood']:>12.2f} "
              f"{res['n_params']:>10d} {res['bic']:>12.2f}")
    
    best_type = min(comparison_results.keys(), 
                   key=lambda k: comparison_results[k]['bic'])
    print(f"\nBest model (by BIC): {best_type}")

test_exercise_1()

## Exercise 2: Three-Asset Portfolio

**Task:** Extend to three assets:
1. Generate 3D synthetic data with regimes
2. Fit 3-state multivariate HMM
3. Visualize 3D correlation structure
4. Compute regime-specific correlation matrices

**Expected Output:** 3-asset HMM with correlation analysis.

In [ ]:
# YOUR CODE HERE
# ---------------

# Generate 3-asset data
def generate_three_asset_data(T=1000):
    # YOUR IMPLEMENTATION
    return None, None

returns_3d, states_3d = generate_three_asset_data(T=1000)

# Fit HMM
hmm_3d = None  # YOUR CODE

# Analyze correlations
correlation_matrices = None  # YOUR CODE

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert returns_3d is not None, "Don't forget to generate data!"
    assert hmm_3d is not None, "Don't forget to fit HMM!"
    assert correlation_matrices is not None, "Don't forget correlations!"
    
    assert returns_3d.shape[1] == 3, "Should have 3 assets"
    assert hmm_3d.D == 3, "HMM should have 3 features"
    assert len(correlation_matrices) == 3, "Should have 3 correlation matrices"
    
    print("✅ Exercise 2 passed!")
    print("\nThree-Asset Portfolio Analysis:")
    print("="*70)
    
    for state in range(3):
        print(f"\nState {state} correlation matrix:")
        print(correlation_matrices[state])

test_exercise_2()

## Exercise 3: Portfolio Optimization by Regime

**Task:** Compute optimal portfolio weights for each regime:
1. Use mean-variance optimization
2. Target risk level: σ_p = 0.15 (annualized)
3. Compare Sharpe ratios across regimes
4. Show regime-dependent allocations

**Expected Output:** Optimal weights and performance by regime.

In [ ]:
# YOUR CODE HERE
# ---------------

def optimize_portfolio_by_regime(hmm, target_vol=0.15):
    """
    Compute optimal weights for each regime.
    
    Parameters
    ----------
    hmm : MultivariateGaussianHMM
        Fitted HMM
    target_vol : float
        Target annualized volatility
    
    Returns
    -------
    weights : ndarray (K, D)
        Optimal weights for each regime
    sharpe_ratios : ndarray (K,)
        Sharpe ratio for each regime
    """
    # YOUR IMPLEMENTATION
    # For each regime:
    #   - Get mean and covariance (annualize: * 252)
    #   - Solve: max w^T μ s.t. sqrt(w^T Σ w) = target_vol
    #   - Compute Sharpe ratio
    
    return None, None

optimal_weights, sharpe_ratios = optimize_portfolio_by_regime(hmm)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert optimal_weights is not None, "Don't forget weights!"
    assert sharpe_ratios is not None, "Don't forget Sharpe ratios!"
    
    assert optimal_weights.shape == (3, 2), "Should be 3 states x 2 assets"
    assert len(sharpe_ratios) == 3, "Should have 3 Sharpe ratios"
    
    # Weights should sum to 1
    assert np.allclose(optimal_weights.sum(axis=1), 1), "Weights should sum to 1"
    
    print("✅ Exercise 3 passed!")
    print("\nRegime-Specific Portfolio Optimization:")
    print("="*70)
    
    for state in range(3):
        print(f"\n{label_map[state]} Regime:")
        print(f"  Asset 1 weight: {optimal_weights[state, 0]:6.2%}")
        print(f"  Asset 2 weight: {optimal_weights[state, 1]:6.2%}")
        print(f"  Sharpe ratio: {sharpe_ratios[state]:6.2f}")

test_exercise_3()

## Summary

### Key Takeaways

1. **Multivariate extension** captures joint dynamics and correlations
2. **Covariance structures** trade off flexibility vs. parameter count
3. **Regime-switching correlations** are critical for portfolio risk
4. **EM algorithm** extends naturally to multivariate case
5. **Model selection** uses BIC to balance fit and complexity

### Covariance Type Guide

| Type | Parameters | Use When |
|------|------------|----------|
| **Full** | K · D(D+1)/2 | Sufficient data, need all correlations |
| **Diagonal** | K · D | Assume conditional independence |
| **Spherical** | K | Very limited data, isotropic noise |
| **Tied** | D(D+1)/2 | Share covariance across states |

### Financial Insights

- **Crisis regimes**: Higher correlations (diversification breaks down)
- **Normal regimes**: Lower correlations (diversification works)
- **Portfolio allocation**: Regime-aware optimization improves risk-adjusted returns
- **Risk management**: Regime-specific VaR and ES

### Practical Considerations

1. **Regularization**: Add small ε to diagonal for numerical stability
2. **Initialization**: Use k-means or random restarts
3. **Convergence**: Monitor log-likelihood, check for local optima
4. **Validation**: Out-of-sample testing, regime stability

### Next Steps

- **Module 4**: Apply to real financial data (S&P 500, VIX)
- **Module 5**: Extensions (hierarchical HMMs, switching AR models)

---